# The Fridge Is Right — Consolidated Capstone Notebook

*Ingredient text → caloric density (kcal/100g). A single-notebook port of Ed Donner's
week 6 "The Price Is Right" pipeline to the food domain.*

**Dataset:** `datahiveai/recipes-with-nutrition` (39,447 recipes, CC BY-NC 4.0)
**Primary task:** regression — unstructured `ingredient_lines` → caloric density (real, measured label)
**Order of play** (mirrors Ed's day1–day5):

| Part | Ed's notebook | Content |
|---|---|---|
| 1 | day1 | Data curation: load, parse, filter, dedup, EDA, splits |
| 2 | day2 | LLM pre-processing: rewrite noisy text into standard summaries |
| 3 | day3 | Evaluation harness, baselines, traditional ML |
| 4 | day4 | Deep neural network + frontier LLMs zero-shot |
| 5 | day5 | Fine-tuning a frontier model |

> **CAUTION:** draft — not executed end-to-end. Column names for the dataset are
> centralized in `COLUMN_MAP` (Part 1); verify them against the inspection cell
> before running anything downstream.


## Part 0 — Setup

Environment variables expected in `.env`: `OPENAI_API_KEY`, `HF_TOKEN` (only if pushing to Hub).
Works with any OpenAI-compatible endpoint (Ollama / LM Studio) — see the `BASE_URL` constant.


In [ ]:
# One-time installs — uncomment on first run
# !pip install -q datasets pandas numpy matplotlib scikit-learn xgboost torch plotly openai python-dotenv tqdm pydantic


In [ ]:
from __future__ import annotations

import json
import math
import random
import re
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from itertools import accumulate
from typing import Callable, Optional, Self

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from tqdm.notebook import tqdm

load_dotenv(override=True)


In [ ]:
# ── Global constants ──────────────────────────────────────────────────────────

SEED: int = 42

# Data source
DATASET_ID: str = "datahiveai/recipes-with-nutrition"

# Prompt template (Ed's PREFIX/QUESTION pattern — completion-style for later QLoRA reuse)
QUESTION: str = "How many calories per 100 grams is this dish?"
PREFIX: str = "Density is "

# Curation bounds
MIN_DENSITY: float = 5.0       # kcal/100g — below this is water/broth or a data error
MAX_DENSITY: float = 900.0     # pure fat ≈ 900 kcal/100g — physical ceiling
MIN_TEXT_CHARS: int = 100      # too short → no signal
MAX_TEXT_CHARS: int = 4_000    # signal plateaus beyond this

# Splits
TRAIN_FRACTION: float = 0.8
VAL_FRACTION: float = 0.1      # test gets the remainder

# OpenAI-compatible endpoint. For local: BASE_URL="http://localhost:11434/v1", key="ollama"
BASE_URL: str | None = None
PREPROCESS_MODEL: str = "gpt-4.1-mini"      # cheap rewriter
FRONTIER_MODEL: str = "gpt-4.1"             # zero-shot benchmark
FINE_TUNE_BASE_MODEL: str = "gpt-4.1-nano-2025-04-14"  # verify current name in OpenAI docs

client = OpenAI(base_url=BASE_URL) if BASE_URL else OpenAI()


## Part 1 — Data Curation *(Ed's day 1)*

Load the dataset, derive the density target, apply curation filters, deduplicate,
inspect distributions, correct skew, and split with a fixed seed.


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_ID, split="train")
print(f"Rows loaded: {len(raw_dataset):,}")


In [ ]:
# Inspect the schema BEFORE trusting COLUMN_MAP below — fix the map, not the code
print(raw_dataset.features)
raw_dataset[0]


In [ ]:
# ── Column map — single point of truth for field names ───────────────────────
# Verify each value against the inspection cell above. Complex fields in this
# dataset are JSON-stringified in the CSV; the loader parses them.

COLUMN_MAP: dict[str, str] = {
    "title": "name",                    # recipe display name
    "ingredients": "ingredient_lines",  # unstructured list of ingredient strings
    "meal_type": "meal_type",           # editorial tag: breakfast/lunch/dinner/snack
    "calories": "calories",             # total kcal for the recipe
    "weight_g": "total_weight_g",       # total recipe weight in grams
}


In [ ]:
class Recipe(BaseModel):
    """A data-point: a dish with a measured caloric density.

    Attributes:
        title: Recipe display name.
        meal_type: Editorial meal tag (breakfast/lunch/dinner/snack).
        density: Caloric density in kcal per 100g — the regression target.
        full: Raw unstructured ingredient text (pre-rewrite).
        summary: LLM-standardized text (set in Part 2, used by all models).
        prompt: Completion-style training prompt (set after summaries exist).
        id: Stable index assigned after curation.
    """

    title: str
    meal_type: str
    density: float
    full: Optional[str] = None
    summary: Optional[str] = None
    prompt: Optional[str] = None
    id: Optional[int] = None

    def make_prompt(self) -> None:
        """Build the completion-style prompt: question + summary + prefixed answer."""
        self.prompt = f"{QUESTION}\n\n{self.summary}\n\n{PREFIX}{round(self.density)}.00"

    def test_prompt(self) -> str:
        """Return the prompt with the answer stripped — for inference."""
        return self.prompt.split(PREFIX)[0] + PREFIX

    def __repr__(self) -> str:
        return f"<{self.title} = {self.density:.0f} kcal/100g>"


In [ ]:
class RecipeLoader:
    """Parses raw dataset rows into curated Recipe objects.

    Applies the curation rules: density bounds, text-length bounds,
    and drops rows with missing or unparseable fields.
    """

    def __init__(self, column_map: dict[str, str]) -> None:
        self._map = column_map

    def load(self, rows) -> list[Recipe]:
        """Parse and filter all rows; returns curated recipes."""
        recipes = [self._parse(row) for row in tqdm(rows)]
        kept = [r for r in recipes if r is not None]
        print(f"Kept {len(kept):,} of {len(rows):,} rows")
        return kept

    def _parse(self, row: dict) -> Recipe | None:
        """Parse one row; None if it fails any curation rule."""
        try:
            ingredients = self._as_text(row[self._map["ingredients"]])
            calories = float(row[self._map["calories"]])
            weight_g = float(row[self._map["weight_g"]])
            meal_type = self._as_meal_type(row[self._map["meal_type"]])
            title = str(row[self._map["title"]]).strip()
        except (KeyError, TypeError, ValueError):
            return None

        if weight_g <= 0 or not title or not meal_type:
            return None
        density = calories / weight_g * 100
        if not (MIN_DENSITY <= density <= MAX_DENSITY):
            return None
        if not (MIN_TEXT_CHARS <= len(ingredients) <= MAX_TEXT_CHARS):
            return None
        return Recipe(title=title, meal_type=meal_type, density=density, full=ingredients)

    @staticmethod
    def _as_text(field) -> str:
        """Normalize a possibly JSON-stringified list field into one text block."""
        if isinstance(field, str) and field.strip().startswith("["):
            field = json.loads(field)
        if isinstance(field, list):
            return "\n".join(str(line).strip() for line in field)
        return str(field).strip()

    @staticmethod
    def _as_meal_type(field) -> str:
        """Normalize meal_type to one lowercase tag; empty string if unusable."""
        if isinstance(field, str) and field.strip().startswith("["):
            field = json.loads(field)
        if isinstance(field, list):
            field = field[0] if field else ""
        tag = str(field).strip().lower()
        # Dataset variants like 'lunch/dinner' — take the first token
        return re.split(r"[/,]", tag)[0].strip()


In [ ]:
loader = RecipeLoader(COLUMN_MAP)
recipes = loader.load(raw_dataset)
recipes[0]


In [ ]:
print(recipes[0].full)


In [ ]:
# ── Deduplication — title AND full text (leakage guard) ──────────────────────
seen: set[tuple[str, str]] = set()
deduped: list[Recipe] = []
for recipe in recipes:
    key = (recipe.title.lower(), recipe.full.lower())
    if key not in seen:
        seen.add(key)
        deduped.append(recipe)
print(f"Removed {len(recipes) - len(deduped):,} duplicates → {len(deduped):,} recipes")
recipes = deduped


In [ ]:
# ── EDA: density distribution ─────────────────────────────────────────────────
densities = [r.density for r in recipes]
plt.figure(figsize=(15, 6))
plt.title(f"Density: avg {np.mean(densities):,.0f}, max {max(densities):,.0f} kcal/100g")
plt.xlabel("kcal per 100g")
plt.ylabel("Count")
plt.hist(densities, rwidth=0.7, color="#4A6FA5", bins=range(0, 920, 20))
plt.show()


In [ ]:
# ── EDA: text lengths and meal-type balance ───────────────────────────────────
lengths = [len(r.full) for r in recipes]
plt.figure(figsize=(15, 4))
plt.title(f"Ingredient text length: avg {np.mean(lengths):,.0f} chars")
plt.hist(lengths, rwidth=0.7, color="#5BA4A4", bins=range(0, 4100, 100))
plt.show()

meal_counts = Counter(r.meal_type for r in recipes)
print(meal_counts.most_common())


In [ ]:
# ── Skew correction (optional) ────────────────────────────────────────────────
# If one meal type or density band dominates, sub-sample it here, then re-plot.
# Iterative: adjust → train → measure. Leaving as pass-through for the first run.
sample: list[Recipe] = recipes


In [ ]:
# ── Shuffle + splits, fixed seed ──────────────────────────────────────────────
random.seed(SEED)
random.shuffle(sample)
for index, recipe in enumerate(sample):
    recipe.id = index

n_train = int(len(sample) * TRAIN_FRACTION)
n_val = int(len(sample) * VAL_FRACTION)
train = sample[:n_train]
val = sample[n_train : n_train + n_val]
test = sample[n_train + n_val :]
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}")


## Part 2 — Data Pre-processing *(Ed's day 2)*

Rewrite noisy ingredient text into a tight standard format with a cheap model.
The **same rewrite must be applied at inference time** — any mismatch degrades performance.

Two paths: synchronous with a thread pool (simple, fine for tens of thousands of rows),
or the Batch API (~50% cheaper — optional cells at the end of this part).
A **no-API fallback** cell lets you skip Part 2 entirely on a first pass.


In [ ]:
SYSTEM_PROMPT: str = (
    "Create a concise description of a dish from its ingredient list. "
    "Respond only in this format:\n"
    "Dish: short precise dish characterization\n"
    "Ingredients: comma-separated core ingredients, no quantities\n"
    "Character: 1 short sentence on richness (fried/creamy/lean/sweet/fresh)"
)


In [ ]:
# Smoke-test the rewrite on one recipe before spending on the full set
response = client.chat.completions.create(
    model=PREPROCESS_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": train[0].full},
    ],
)
print(train[0].full)
print("---")
print(response.choices[0].message.content)


In [ ]:
class SummaryRewriter:
    """Rewrites raw ingredient text into the standard summary format.

    Attributes:
        model: OpenAI-compatible model name used for the rewrite.
        workers: Thread concurrency for the synchronous path.
    """

    def __init__(self, model: str = PREPROCESS_MODEL, workers: int = 8) -> None:
        self.model = model
        self.workers = workers

    def rewrite(self, recipe: Recipe) -> None:
        """Set recipe.summary from the model rewrite; falls back to raw text on error."""
        try:
            response = client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": recipe.full},
                ],
            )
            recipe.summary = response.choices[0].message.content
        except Exception as error:
            print(f"Rewrite failed for id={recipe.id}: {error}")
            recipe.summary = recipe.full

    def run(self, recipes: list[Recipe]) -> None:
        """Rewrite all recipes concurrently."""
        with ThreadPoolExecutor(max_workers=self.workers) as pool:
            list(tqdm(pool.map(self.rewrite, recipes), total=len(recipes)))


In [ ]:
# COST GATE: estimate on a small slice first, then scale up deliberately.
# rewriter = SummaryRewriter()
# rewriter.run(train[:1000])

# NO-API FALLBACK — skip the rewrite entirely on a first pass:
for recipe in train + val + test:
    if recipe.summary is None:
        recipe.summary = recipe.full


In [ ]:
# ── OPTIONAL: Batch API path (~50% cost reduction) ────────────────────────────
def make_batch_line(recipe: Recipe) -> str:
    """Return one Batch-API JSONL line for a recipe rewrite request."""
    body = {
        "model": PREPROCESS_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": recipe.full},
        ],
    }
    wrapper = {
        "custom_id": str(recipe.id),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": body,
    }
    return json.dumps(wrapper)


def write_batch_file(recipes: list[Recipe], filename: str) -> None:
    """Write Batch-API input JSONL for the given recipes."""
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(make_batch_line(r) for r in recipes))


In [ ]:
# OPTIONAL batch execution — uncomment to run
# write_batch_file(train[:1000], "batch_0_1000.jsonl")
# with open("batch_0_1000.jsonl", "rb") as f:
#     batch_input = client.files.create(file=f, purpose="batch")
# batch = client.batches.create(
#     input_file_id=batch_input.id,
#     endpoint="/v1/chat/completions",
#     completion_window="24h",
# )
# # Poll: client.batches.retrieve(batch.id).status → 'completed'
# # result_file = client.batches.retrieve(batch.id).output_file_id
# # content = client.files.content(result_file)
# # Match results by custom_id — results are OUT OF ORDER, never match by line position


In [ ]:
# ── Build prompts once summaries exist ────────────────────────────────────────
for recipe in train + val + test:
    recipe.make_prompt()
print(train[0].prompt)
print("---")
print(test[0].test_prompt())


## Part 3 — Evaluation Harness, Baselines, Traditional ML *(Ed's day 3)*

One harness for every model: **MAE ± 95% CI** on the same held-out test slice,
color-coded scatter, and error-trend chart. Then climb the ladder:
random → constant → feature LR → bag-of-words LR → Random Forest → XGBoost.


In [ ]:
GREEN, YELLOW, RED, RESET = "\033[92m", "\033[93m", "\033[91m", "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

DEFAULT_SIZE: int = 200
DEFAULT_WORKERS: int = 5


class Tester:
    """Runs a predictor against labeled test data and reports MAE, scatter, CI.

    Attributes:
        predictor: Callable Recipe -> float | str (strings are parsed).
        data: Labeled test set.
        title: Display title, derived from the predictor name by default.
        size: Number of test items to evaluate.
        workers: Thread concurrency (set 1 if hitting API rate limits).
    """

    def __init__(
        self,
        predictor: Callable,
        data: list[Recipe],
        title: str | None = None,
        size: int = DEFAULT_SIZE,
        workers: int = DEFAULT_WORKERS,
    ) -> None:
        self.predictor = predictor
        self.data = data
        self.title = title or predictor.__name__.replace("_", " ").title()
        self.size = min(size, len(data))
        self.workers = workers
        self.guesses: list[float] = []
        self.truths: list[float] = []
        self.errors: list[float] = []
        self.colors: list[str] = []
        self.titles: list[str] = []

    @staticmethod
    def post_process(value: float | str) -> float:
        """Extract a float from a possibly-noisy string prediction."""
        if isinstance(value, str):
            cleaned = value.replace(",", "").replace("kcal", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", cleaned)
            return float(match.group()) if match else 0.0
        return float(value)

    @staticmethod
    def color_for(error: float, truth: float) -> str:
        """Green/orange/red banding tuned for the 0-900 kcal/100g scale."""
        if error < 25 or error / truth < 0.15:
            return "green"
        if error < 75 or error / truth < 0.35:
            return "orange"
        return "red"

    def run_datapoint(self, index: int) -> None:
        """Predict one item, record error and color."""
        recipe = self.data[index]
        guess = self.post_process(self.predictor(recipe))
        truth = recipe.density
        error = abs(guess - truth)
        color = self.color_for(error, truth)
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.colors.append(color)
        self.titles.append(recipe.title[:40])
        print(
            f"{COLOR_MAP[color]}{index + 1}: Guess {guess:.0f} "
            f"Truth {truth:.0f} Error {error:.0f}{RESET}"
        )

    def run(self) -> None:
        """Evaluate, then render the scatter and the error-trend chart."""
        with ThreadPoolExecutor(max_workers=self.workers) as pool:
            list(pool.map(self.run_datapoint, range(self.size)))
        self.report()

    def report(self) -> None:
        """Print MAE with 95% CI and show both charts."""
        mae = float(np.mean(self.errors))
        ci = 1.96 * float(np.std(self.errors)) / math.sqrt(len(self.errors))
        hits = sum(1 for c in self.colors if c == "green")
        print(f"\n{self.title}  MAE: {mae:.1f} ± {ci:.1f} kcal/100g  "
              f"Hits: {hits / len(self.errors):.0%}")
        self._scatter(mae, ci)
        self._error_trend()

    def _scatter(self, mae: float, ci: float) -> None:
        max_val = max(max(self.truths), max(self.guesses))
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val],
                                 mode="lines", line=dict(color="#8A9BB0", dash="dash"),
                                 showlegend=False))
        fig.add_trace(go.Scatter(x=self.truths, y=self.guesses, mode="markers",
                                 marker=dict(color=self.colors, size=6, opacity=0.7),
                                 text=self.titles, showlegend=False))
        fig.update_layout(title=f"{self.title}  MAE: {mae:.1f} ± {ci:.1f}",
                          xaxis_title="Actual kcal/100g", yaxis_title="Predicted",
                          width=800, height=500, template="plotly_white")
        fig.show()

    def _error_trend(self) -> None:
        n = len(self.errors)
        x = list(range(1, n + 1))
        running_means = [s / i for s, i in zip(accumulate(self.errors), x)]
        fig = go.Figure(go.Scatter(x=x, y=running_means, mode="lines",
                                   line=dict(width=3, color="#C47E7E")))
        fig.update_layout(title=f"{self.title} — cumulative average error",
                          xaxis_title="Test items", yaxis_title="Cumulative MAE",
                          width=800, height=300, template="plotly_white")
        fig.show()


def evaluate(predictor: Callable, data: list[Recipe], size: int = DEFAULT_SIZE,
             workers: int = DEFAULT_WORKERS) -> None:
    """Convenience wrapper: build a Tester and run it."""
    Tester(predictor, data, size=size, workers=workers).run()


In [ ]:
# ── Baseline 1: random ────────────────────────────────────────────────────────
def random_density(recipe: Recipe) -> float:
    """Uniform random guess across the physical density range."""
    return random.uniform(MIN_DENSITY, MAX_DENSITY)

random.seed(SEED)
evaluate(random_density, test)


In [ ]:
# ── Baseline 2: constant (train mean) ─────────────────────────────────────────
TRAIN_MEAN_DENSITY: float = float(np.mean([r.density for r in train]))

def constant_density(recipe: Recipe) -> float:
    """Always predict the training-set mean density."""
    return TRAIN_MEAN_DENSITY

evaluate(constant_density, test)


In [ ]:
# ── Simple engineered features + linear regression ────────────────────────────
from sklearn.linear_model import LinearRegression

RICH_TERMS = ["butter", "cream", "oil", "cheese", "sugar", "chocolate", "bacon"]
LEAN_TERMS = ["lettuce", "tomato", "broth", "cucumber", "spinach", "zucchini"]


def get_features(recipe: Recipe) -> dict[str, float]:
    """Hand-engineered features from the summary text."""
    text = recipe.summary.lower()
    return {
        "text_length": len(text),
        "n_lines": text.count("\n") + text.count(",") + 1,
        "rich_hits": sum(text.count(term) for term in RICH_TERMS),
        "lean_hits": sum(text.count(term) for term in LEAN_TERMS),
    }


def to_frame(recipes: list[Recipe]) -> pd.DataFrame:
    frame = pd.DataFrame([get_features(r) for r in recipes])
    frame["density"] = [r.density for r in recipes]
    return frame


np.random.seed(SEED)
train_frame, test_frame = to_frame(train), to_frame(test)
feature_columns = ["text_length", "n_lines", "rich_hits", "lean_hits"]
feature_lr = LinearRegression().fit(train_frame[feature_columns], train_frame["density"])
for column, coef in zip(feature_columns, feature_lr.coef_):
    print(f"{column}: {coef:.3f}")


def feature_linear_regression(recipe: Recipe) -> float:
    """Predict density from the four engineered features."""
    row = pd.DataFrame([get_features(recipe)])
    return float(feature_lr.predict(row[feature_columns])[0])

evaluate(feature_linear_regression, test)


In [ ]:
# ── Bag of words + linear regression ──────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

np.random.seed(SEED)
train_targets = np.array([r.density for r in train])
train_documents = [r.summary for r in train]

vectorizer = CountVectorizer(max_features=2000, stop_words="english")
train_matrix = vectorizer.fit_transform(train_documents)
bow_lr = LinearRegression().fit(train_matrix, train_targets)


def bow_linear_regression(recipe: Recipe) -> float:
    """Predict density from a bag-of-words linear model."""
    return float(bow_lr.predict(vectorizer.transform([recipe.summary]))[0])

evaluate(bow_linear_regression, test)


In [ ]:
# ── Random Forest (on a subset — full set is slow) ────────────────────────────
from sklearn.ensemble import RandomForestRegressor

RF_SUBSET: int = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=4)
rf_model.fit(train_matrix[:RF_SUBSET], train_targets[:RF_SUBSET])


def random_forest(recipe: Recipe) -> float:
    """Predict density with the Random Forest over bag-of-words features."""
    return float(rf_model.predict(vectorizer.transform([recipe.summary]))[0])

evaluate(random_forest, test)


In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
import xgboost as xgb

np.random.seed(SEED)
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, random_state=SEED, n_jobs=4,
    learning_rate=0.08, max_depth=6,
)
xgb_model.fit(train_matrix, train_targets)


def xg_boost(recipe: Recipe) -> float:
    """Predict density with XGBoost over bag-of-words features."""
    return float(xgb_model.predict(vectorizer.transform([recipe.summary]))[0])

evaluate(xg_boost, test)


## Part 4 — Deep Neural Network + Frontier LLMs *(Ed's day 4)*

A residual bag-of-words DNN with a **log-normalized target** (density is right-skewed),
then frontier models zero-shot. Per the course's own finding: expect frontier zero-shot
to be very competitive — food is a domain these models know deeply.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import HashingVectorizer
from torch.utils.data import DataLoader, TensorDataset

# ── DNN constants ─────────────────────────────────────────────────────────────
N_FEATURES: int = 5_000
HIDDEN_SIZE: int = 512
DROPOUT_PROB: float = 0.2
BATCH_SIZE: int = 64
LEARNING_RATE: float = 1e-3
EPOCHS: int = 5
MAX_GRAD_NORM: float = 1.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

hashing_vectorizer = HashingVectorizer(
    n_features=N_FEATURES, stop_words="english", binary=True
)
X_train = hashing_vectorizer.transform([r.summary for r in train])
y_raw = np.array([r.density for r in train], dtype=np.float32)

# Log-normalize the skewed target; constants ship as code — regenerate on retrain
y_log = np.log1p(y_raw)
Y_MEAN: float = float(y_log.mean())
Y_STD: float = float(y_log.std())
y_train = (y_log - Y_MEAN) / Y_STD
print(f"Y_MEAN={Y_MEAN:.4f}  Y_STD={Y_STD:.4f}")


In [ ]:
class ResidualBlock(nn.Module):
    """Two-layer block with LayerNorm, dropout, and a skip connection."""

    def __init__(self, hidden_size: int, dropout: float) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )
        self.relu = nn.ReLU()

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.relu(self.block(inputs) + inputs)


class DensityNet(nn.Module):
    """Bag-of-words regressor: projection → 2 residual blocks → scalar."""

    def __init__(self, input_size: int = N_FEATURES, hidden: int = HIDDEN_SIZE) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden),
            nn.ReLU(),
            ResidualBlock(hidden, DROPOUT_PROB),
            ResidualBlock(hidden, DROPOUT_PROB),
            nn.Linear(hidden, 1),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.network(inputs)


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
torch.manual_seed(SEED)
model = DensityNet().to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

dataset = TensorDataset(
    torch.FloatTensor(X_train.toarray()),
    torch.FloatTensor(y_train).unsqueeze(1),
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

loss_function = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for batch_x, batch_y in tqdm(loader, desc=f"Epoch {epoch + 1}"):
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_function(model(batch_x), batch_y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch + 1}: loss {epoch_loss / len(loader):.4f}")


In [ ]:
def neural_network(recipe: Recipe) -> float:
    """Predict density with the DNN, inverting the log-normalization."""
    model.eval()
    features = hashing_vectorizer.transform([recipe.summary]).toarray()
    with torch.no_grad():
        normalized = model(torch.FloatTensor(features).to(DEVICE)).item()
    return float(np.expm1(normalized * Y_STD + Y_MEAN))

evaluate(neural_network, test)


In [ ]:
# ── Frontier LLMs, zero-shot ──────────────────────────────────────────────────
def messages_for(recipe: Recipe) -> list[dict[str, str]]:
    """Zero-shot prompt: ask for kcal/100g, answer only."""
    instruction = (
        "Estimate the caloric density of this dish in kcal per 100 grams. "
        "Respond only with the number, no explanation."
    )
    return [{"role": "user", "content": f"{instruction}\n\n{recipe.summary}"}]


def frontier_zero_shot(recipe: Recipe) -> str:
    """Zero-shot density estimate from the frontier model."""
    response = client.chat.completions.create(
        model=FRONTIER_MODEL, messages=messages_for(recipe), max_tokens=10
    )
    return response.choices[0].message.content


In [ ]:
print(test[0].summary)
print(frontier_zero_shot(test[0]), "vs truth", round(test[0].density))


In [ ]:
evaluate(frontier_zero_shot, test)
# Add more providers by swapping FRONTIER_MODEL / BASE_URL — e.g. a local
# Ollama model for a free comparison point.


## Part 5 — Fine-Tuning a Frontier Model *(Ed's day 5)*

OpenAI SFT: format JSONL → upload → train → evaluate. Guidance from the course:
50–200 train examples, 1 epoch. **Never include the assistant turn in evaluation
prompts** — that returns your provided value and scores a spurious perfect result.


In [ ]:
FINE_TUNE_TRAIN_SIZE: int = 200
FINE_TUNE_VAL_SIZE: int = 50

fine_tune_train = train[:FINE_TUNE_TRAIN_SIZE]
fine_tune_validation = val[:FINE_TUNE_VAL_SIZE]


In [ ]:
def training_messages_for(recipe: Recipe) -> list[dict[str, str]]:
    """User prompt + expected assistant answer — TRAINING JSONL ONLY."""
    instruction = (
        "Estimate the caloric density of this dish in kcal per 100 grams. "
        "Respond only with the number, no explanation."
    )
    return [
        {"role": "user", "content": f"{instruction}\n\n{recipe.summary}"},
        {"role": "assistant", "content": f"{round(recipe.density)}"},
    ]


def write_jsonl(recipes: list[Recipe], filename: str) -> None:
    """Write fine-tuning JSONL: one messages object per line."""
    with open(filename, "w", encoding="utf-8") as f:
        for recipe in recipes:
            f.write(json.dumps({"messages": training_messages_for(recipe)}) + "\n")


write_jsonl(fine_tune_train, "fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "fine_tune_validation.jsonl")


In [ ]:
with open("fine_tune_train.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with open("fine_tune_validation.jsonl", "rb") as f:
    validation_file = client.files.create(file=f, purpose="fine-tune")
train_file, validation_file


In [ ]:
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model=FINE_TUNE_BASE_MODEL,
    seed=SEED,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="fridge-density",
)
job.id


In [ ]:
# Poll until status == 'succeeded' (validating_files → running → succeeded)
client.fine_tuning.jobs.retrieve(job.id)


In [ ]:
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=10).data


In [ ]:
fine_tuned_model_name = client.fine_tuning.jobs.retrieve(job.id).fine_tuned_model
fine_tuned_model_name


In [ ]:
def fine_tuned_frontier(recipe: Recipe) -> str:
    """Density estimate from the fine-tuned model — NO assistant turn in the prompt."""
    response = client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=messages_for(recipe),   # inference messages only
        max_tokens=7,
    )
    return response.choices[0].message.content


In [ ]:
print(round(test[0].density))
print(fine_tuned_frontier(test[0]))


In [ ]:
evaluate(fine_tuned_frontier, test)


## Results Log

Fill in as each model completes — this table becomes the headline chart.

| # | Model | MAE (kcal/100g) | ± 95% CI | Notes |
|---|---|---|---|---|
| 1 | Random | | | floor |
| 2 | Constant (train mean) | | | floor |
| 3 | Feature LR | | | |
| 4 | Bag-of-words LR | | | |
| 5 | Random Forest | | | 15K subset |
| 6 | XGBoost | | | honest classical benchmark |
| 7 | DNN (residual, log-norm) | | | |
| 8 | Frontier zero-shot | | | may win — that's the story |
| 9 | Frontier fine-tuned | | | may underperform #8 |

**Next steps:** QLoRA fine-tune of Llama 3.2 3B (density is 0–900 → single-token
targets), then the agent phase: Chroma RAG agent, Modal specialist, ensemble,
candidate-generating planner, Gradio app.
